# TP 2 - Préparation de la base RAG


Ce notebook prépare la base documentaire du RAG.
On construit ici une version de base (V1) comme point de référence.

### 0.1. Objectif
- **TP 2_1** : Créer une base de données vectorielle (`chroma_db_rag_v1`) à partir de documents Markdown issus de guides de voyage
- **TP 2_2** : Créer un assistant de voyage RAG simple
- **TP 2_3** : Créer une base de données mieux structurée / optimisée (`chroma_db_rag_v2`)
- **TP 2_4** : Créer un assistant de voyage RAG avec des méthodes avancées de retrieval

### 0.2. Documentation générale

[Google GenAI Python SDK](https://googleapis.github.io/python-genai/)

[ChromaDB](https://docs.trychroma.com/)

In [ ]:
from pathlib import Path

import chromadb
from tqdm import tqdm

from shared.config import ROOT_DIR, genai_client, project_settings
from shared.rag_utils import (
    CHROMA_MAX_BATCH_SIZE,
    MarkdownDocument,
    RAGChunk,
    rag_describe_chunks,
    rag_load_markdown_documents,
)

DATA_DIR = ROOT_DIR / "TP2_travel_planner_RAG" / "data"
MARKDOWN_DIR = DATA_DIR / "guides_markdown"
CHROMA_DIR_V1 = DATA_DIR / "chroma_db_rag_v1"

### 0.3. Récapitulatif des fonctions utilisées dans ce notebook

**Fournies**

- `rag_load_markdown_documents` : charge les fichiers `.md` d'un dossier en `MarkdownDocument`
- `rag_describe_chunks` : affiche des statistiques et la distribution des tailles de chunks
- `CHROMA_MAX_BATCH_SIZE` : taille maximale d'un batch d'ajout Chroma

--> Disponibles dans `shared/rag_utils.py`

**À coder dans ce notebook**

- `rag_chunk_document_by_chars` : découpe un document en chunks de taille fixe avec overlap
- `rag_embed_text_batch` : calcule les embeddings d'une liste de textes
- `rag_embed_all_chunks` : calcule les embeddings de tous les chunks, en appelant `rag_embed_text_batch` par groupes
- `rag_index_chunks_chroma` : indexe des chunks vectorisés dans Chroma

--> À implémenter ici, puis à copier dans `shared/rag_utils.py`

### 0.4. Charger les documents

In [ ]:
documents = rag_load_markdown_documents(MARKDOWN_DIR)

total_characters = sum(len(doc["text"]) for doc in documents)
print(f"Documents Markdown chargés : {len(documents)}")
print(f"Nombre total de caractères : {total_characters}")

---
## 1. Stratégie de chunking classique (V1)

Un document complet est **trop long** pour la recherche par vecteurs latents.
On le **découpe en chunks** de taille comparable, mesurés en nombre de caractères.
L'**overlap** garde une zone de texte partagée entre deux chunks voisins pour conserver la continuité.

Vous pouvez **expérimenter** avec les paramètres `chunk_chars` et `chunk_overlap_chars` pour observer l'effet sur le nombre de chunks et la qualité du **retrieval** (par la suite, dans le notebook `2_2`).

Repère attendu : entre 100 et 500 chunks au total.

### 1.1. Coder la fonction de chunking

In [ ]:
def rag_chunk_document_by_chars(
    document: MarkdownDocument,
    chunk_chars: int,
    chunk_overlap_chars: int,
) -> list[RAGChunk]:
    """Découper un document en chunks de taille fixe avec fenêtre glissante

    Entrées
    - document : un `MarkdownDocument`
    - chunk_chars : taille max d'un chunk en caractères
    - chunk_overlap_chars : chevauchement entre deux chunks successifs

    Sortie
    - liste de `RAGChunk`
    """
    chunks: list[RAGChunk] = []
    ...
    return chunks

### 1.2. Appliquer le chunking à tous les documents

In [ ]:
chunks_v1 = []
...
print(f"Nombre total de chunks V1 : {len(chunks_v1)}")

`rag_describe_chunks` affiche des statistiques (nombre de chunks, taille min/max/moyenne) et un histogramme de la distribution des tailles par document source.

C'est de la visualisation pure (matplotlib), sans rapport avec une API GenAI : la fonction est fournie toute faite dans `shared/rag_utils.py`.

### 1.3. Inspecter les chunks (statistiques)

In [ ]:
rag_describe_chunks(chunks_v1)

### 1.4. Afficher quelques chunks

Afficher quelques chunks pour comprendre le résultat du découpage.

In [ ]:
...

---
## 2. Embeddings et indexation

Dans cette partie, nous allons coder le calcul des embeddings et leur indexation dans une base de données Chroma.

**Étape 1 : Calcul des embeddings** - Gemini API

- `rag_embed_text_batch` calcule les embeddings d'un lot (*batch*) de textes (l'API a une limite de textes à envoyer en un seul appel)
- `rag_embed_all_chunks` appelle `rag_embed_text_batch` autant de fois que nécessaire pour calculer les embeddings de tous les chunks (par batch)

**Étape 2 : Indexation des embeddings** - Chroma

- `rag_index_chunks_chroma` indexe les embeddings dans une base de données Chroma (SQLite)

### 2.1. Coder la fonction d'embedding

In [ ]:
def rag_embed_text_batch(texts: list[str]) -> list[list[float]]:
    """Calculer les embeddings d'une liste de textes via l'API Google GenAI

    Entrées
    - texts : liste de chaînes à vectoriser

    Sortie
    - liste de vecteurs `list[float]` dans le même ordre que `texts`
    """
    # DOC (embed_content) : https://ai.google.dev/gemini-api/docs/embeddings
    ...
    return ...

### 2.2. Coder la fonction de batch embedding

In [ ]:
def rag_embed_all_chunks(chunks: list[RAGChunk], batch_size: int = 16) -> list[RAGChunk]:
    """Calculer les embeddings de tous les chunks par batch

    Entrées
    - chunks : liste de `RAGChunk`
    - batch_size : nombre de chunks traités par appel embedding

    Sortie
    - liste de `RAGChunk` avec `embedding` renseigné
    """
    embedded_chunks: list[RAGChunk] = []
    ...
    return embedded_chunks

### 2.3. Calculer les embeddings des chunks

In [ ]:
chunk_embeddings_v1 = rag_embed_all_chunks(
    chunks=chunks_v1,
    batch_size=16
)

### 2.4. Coder la fonction d'indexation

In [ ]:
def rag_index_chunks_chroma(persist_dir: Path, chunks: list[RAGChunk]) -> None:
    """Indexer des chunks vectorisés dans Chroma

    Entrées
    - persist_dir : dossier de persistance Chroma
    - chunks : liste de `RAGChunk` avec embeddings calculés
    """
    # DOC (PersistentClient) : https://docs.trychroma.com/docs/run-chroma/clients
    # DOC (get_or_create_collection) : https://docs.trychroma.com/docs/collections/manage-collections
    client = ...
    collection = ...
    # DOC (add) : https://docs.trychroma.com/docs/collections/add-data
    ...

### 2.5. Indexer les chunks dans Chroma

In [ ]:
rag_index_chunks_chroma(
    persist_dir=CHROMA_DIR_V1,
    chunks=chunk_embeddings_v1
)

print(f"V1 - Chunks indexés : {len(chunk_embeddings_v1)}")
print(f"V1 - Dimension des embeddings : {len(chunk_embeddings_v1[0].embedding)}")

---
## Déplacer vers shared/

Les fonctions codées dans ce notebook sont utilisées dans les notebooks suivants.

Copiez-les dans `shared/rag_utils.py` avant de passer à `2_2_rag_assistant_simple.ipynb` :

- `rag_chunk_document_by_chars`
- `rag_embed_text_batch`
- `rag_embed_all_chunks`
- `rag_index_chunks_chroma`